# Phase 3 — document complet d’analyse

## Partie A — audit des artefacts préparés

Cette première partie reprend les contrôles descriptifs du notebook initial : intégrité, populations figées, entonnoir, scénarios, couverture, contingence et faisabilité.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'config' / 'pipeline.json').is_file():
            return candidate
    raise FileNotFoundError('Racine du projet introuvable depuis le notebook')

PROJECT_ROOT = find_project_root()
ANALYSIS_DIR = PROJECT_ROOT / 'outputs' / 'moteur' / 'phase3' / 'analysis'
FILES = {
    'full': 'phase3_analysis_full.csv',
    'covered': 'phase3_analysis_covered.csv',
    'primary': 'phase3_analysis_primary.csv',
    'selection': 'phase3_selection_summary.csv',
    'scenarios': 'phase3_scenario_summary.csv',
    'contingency': 'phase3_contingency_top1.csv',
    'integrity': 'phase3_integrity_checks.csv',
    'feasibility': 'phase3_modeling_feasibility.csv',
    'separation': 'phase3_separation_diagnostics.csv',
}
tables = {name: pd.read_csv(ANALYSIS_DIR / filename) for name, filename in FILES.items()}
{name: frame.shape for name, frame in tables.items()}

## 1. Intégrité et populations figées

Trois populations sont distinguées :

- **Jeu complet (3 761 images)** : toutes les images, utilisé pour mesurer la couverture globale et les limites structurelles du dispositif.
- **Jeu couvert (1 725 images)** : images dont la classe réelle est couverte par les BSV. Il sert à étudier la décidabilité du moteur hors défaut structurel de couverture.
- **Jeu principal (1 246 images)** : classe réelle couverte, scénario réalisable et verdict Top-1 `Coherent` ou `Non_corrobore`. C'est la population destinée à l'analyse statistique principale.

Les classes non couvertes restent dans le jeu complet, mais sont exclues du test statistique principal.

In [ ]:
integrity = tables['integrity']
assert integrity['passed'].astype(bool).all(), integrity.loc[~integrity['passed'].astype(bool)]
display(integrity)
populations = pd.DataFrame({
    'population': ['Jeu complet', 'Jeu couvert', 'Jeu principal'],
    'n': [len(tables['full']), len(tables['covered']), len(tables['primary'])],
})
display(populations)
assert populations.set_index('population').loc['Jeu complet', 'n'] == 3761
assert populations.set_index('population').loc['Jeu couvert', 'n'] == 1725
assert populations.set_index('population').loc['Jeu principal', 'n'] == 1246

## 2. Entonnoir et causes d’exclusion

In [ ]:
selection = tables['selection']
display(selection)
selection_counts = selection.set_index('etape')['nb_images']
recomposition = (
    selection_counts['exclusion_classe_non_couverte']
    + selection_counts['exclusion_scenario_non_realisable']
    + selection_counts['exclusion_sans_bulletin']
    + selection_counts['exclusion_hors_periode_litterature']
    + selection_counts['exclusion_autre_indetermine']
    + selection_counts['jeu_principal']
)
assert recomposition == selection_counts['jeu_complet'] == 3761
print(
    f"Recomposition vérifiée : {selection_counts['exclusion_classe_non_couverte']} + "
    f"{selection_counts['exclusion_scenario_non_realisable']} + "
    f"{selection_counts['exclusion_sans_bulletin']} + "
    f"{selection_counts['exclusion_hors_periode_litterature']} + "
    f"{selection_counts['exclusion_autre_indetermine']} + "
    f"{selection_counts['jeu_principal']} = {recomposition} images."
)
plot = selection.set_index('etape')['nb_images'].plot.barh(figsize=(9, 5), color='#4C78A8')
plot.set(title='Entonnoir de sélection V3', xlabel='Effectif images', ylabel='')
plt.tight_layout()

## 3. Scénarios demandés, réalisés et faisabilité

In [ ]:
scenarios = tables['scenarios']
global_scenarios = scenarios.query("scope == 'global'").copy()
display(global_scenarios)
ax = global_scenarios.set_index('scenario_assigne')[['nb_demande', 'nb_realisable']].plot.bar(
    figsize=(8, 4), color=['#9ECAE1', '#2171B5']
)
ax.set(title='Scénarios demandés et réalisables', xlabel='', ylabel='Effectif images')
plt.xticks(rotation=0)
plt.tight_layout()
display(scenarios.query("scope == 'classe'").pivot(
    index='scope_value', columns='scenario_assigne', values='taux_realisation'
).style.format('{:.1%}'))

## 4. Couverture, classes, verdicts et scores

In [ ]:
full = tables['full']
primary = tables['primary']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
full.groupby(['true_label', 'classe_couverte_bsv']).size().unstack(fill_value=0).plot.barh(ax=axes[0])
axes[0].set(title='Couverture BSV par classe réelle', xlabel='Effectif images', ylabel='')
primary['verdict_bsv_top1'].value_counts().plot.bar(ax=axes[1], color=['#F28E2B', '#59A14F'])
axes[1].set(title='Verdicts dans le jeu principal', xlabel='', ylabel='Effectif images')
primary['score_cnn_top1'].plot.hist(ax=axes[2], bins=30, color='#4C78A8')
axes[2].set(title='Scores CNN Top-1', xlabel='Score', ylabel='Effectif images')
plt.tight_layout()

## 5. Contingence exploratoire

In [ ]:
contingency = tables['contingency']
display(contingency)
display(contingency.pivot(index='cnn_correct', columns='verdict_bsv_top1', values='observed'))
first = contingency.iloc[0]
print(f"Test: {first['test']} — statistique={first['statistic']:.4f}, p={first['p_value']:.4g}")
print(f"Effectif attendu minimal: {contingency['expected'].min():.2f}")

## 6. Faisabilité de la modélisation et séparation

In [ ]:
display(tables['feasibility'])
separation = tables['separation']
display(separation.query('separation or quasi_separation'))

## 7. Résumé descriptif

Les proportions de scénarios sont imposées par le plan expérimental et ne représentent pas leur fréquence naturelle. La contingence est exploratoire : elle ne démontre pas l'apport incrémental du BSV au-delà du score CNN. Les avertissements de séparation et le nombre d'événements par paramètre doivent guider la spécification des modèles M0/M1.

---

# Partie B — analyse scientifique complète

Les cellules suivantes complètent l’audit précédent et recalculent directement les résultats scientifiques à partir des données préparées.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import chi2, chi2_contingency
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
    f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score,
    average_precision_score, log_loss, brier_score_loss)
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from patsy import dmatrix

warnings.filterwarnings("ignore", category=RuntimeWarning)
SEED=2026; BOOTSTRAP_REPLICATIONS=500; CV_FOLDS=5
ROOT = PROJECT_ROOT
PATHS={
 "full":ROOT/"outputs/moteur/phase3/analysis/phase3_analysis_full.csv",
 "assignments":ROOT/"data/processed/moteur/phase3/phase3_scenario_assignments.csv",
 "query":ROOT/"outputs/moteur/phase3/requete_phase3.json",
 "predictions":ROOT/"outputs/moteur/phase3/cnn_topk_predictions.json",
 "verdicts":ROOT/"data/processed/moteur/bsv_compatibility_phase3.csv",
 "coverage":ROOT/"data/raw/moteur/referentiel_cnn_bsv.csv",
}
for path in PATHS.values(): assert path.exists(), path

## 1. Chargement des données préparées

In [ ]:
jeu_source=pd.read_csv(PATHS["full"])
assignments=pd.read_csv(PATHS["assignments"])
verdicts=pd.read_csv(PATHS["verdicts"])
coverage=pd.read_csv(PATHS["coverage"], sep=None, engine="python")
query=json.loads(PATHS["query"].read_text(encoding="utf-8"))
predictions=json.loads(PATHS["predictions"].read_text(encoding="utf-8"))
query_df=pd.DataFrame(query)
predictions_df=pd.DataFrame(predictions)
loaded={"Top-1":jeu_source,"assignations":assignments,"requête aveugle":query_df,
        "prédictions CNN":predictions_df,"verdicts moteur":verdicts,"référentiel":coverage}
display(pd.DataFrame([{"source":k,"lignes":len(v),"colonnes":len(v.columns)} for k,v in loaded.items()]))
for name,frame in loaded.items():
 print(f"\n{name}: {list(frame.columns)}"); display(frame.head(2))

## 2. Contrôles d’intégrité

Chaque contrôle affiche `OK`; toute violation interrompt immédiatement le notebook avec une erreur explicite.

In [ ]:
checks=[]
def check(name, condition, detail=""):
 assert bool(condition), f"ERREUR — {name}: {detail}"
 checks.append({"contrôle":name,"statut":"OK","détail":detail})

query_ids=query_df.image_id.astype(str); pred_ids=predictions_df.observation_id.astype(str)
full_ids=jeu_source.image_id.astype(str); verdict_ids=verdicts.image_id.astype(str)
check("3 761 images uniques",len(jeu_source)==3761 and full_ids.nunique()==3761,f"n={len(jeu_source)}")
check("une ligne Top-1 par image",not full_ids.duplicated().any())
check("aucun identifiant perdu ou ajouté",set(full_ids)==set(query_ids)==set(pred_ids)==set(verdict_ids))
check("absence de doublons Top-1",not jeu_source.image_id.duplicated().any())
qtop=pd.DataFrame([{"image_id":str(x["image_id"]),"q_class":x["top_k"][0]["classe"],"q_score":float(x["top_k"][0]["score"])} for x in query])
ptop=pd.DataFrame([{"image_id":str(x["observation_id"]),"p_rank":x["predictions"][0]["rank"],"p_class":x["predictions"][0]["label"],"p_score":float(x["predictions"][0]["score"])} for x in predictions])
coh=qtop.merge(ptop,on="image_id").merge(jeu_source[["image_id","classe_cnn_top1","score_cnn_top1"]],on="image_id")
check("rang Top-1 déclaré",(coh.p_rank==1).all())
check("classes requête/prédictions/Top-1",(coh.q_class==coh.p_class).all() and (coh.q_class==coh.classe_cnn_top1).all())
check("scores requête/prédictions/Top-1",np.allclose(coh.q_score,coh.p_score,atol=1e-9) and np.allclose(coh.q_score,coh.score_cnn_top1,atol=1e-9))
vtop=verdicts.merge(qtop,left_on=["image_id","classe_cnn_predite"],right_on=["image_id","q_class"])
vcheck=jeu_source.merge(vtop[["image_id","compatibilite"]],on="image_id")
check("verdicts moteur cohérents",(vcheck.verdict_bsv_top1==vcheck.compatibilite).all())
forbidden={"true_label","scenario_assigne","scenario_realise","scenario_realisable","seed"}
query_keys=set().union(*(set(x) for x in query)); topk_keys=set().union(*(set(y) for x in query for y in x["top_k"]))
check("aucune fuite vers le moteur",not ((query_keys|topk_keys)&forbidden),str(sorted((query_keys|topk_keys)&forbidden)))
check("scores dans [0,1]",jeu_source.score_cnn_top1.between(0,1).all())
check("verdicts autorisés",set(jeu_source.verdict_bsv_top1.dropna()) <= {"Coherent","Non_corrobore","Indetermine"})
check("scénarios autorisés",set(jeu_source.scenario_assigne.dropna()) <= {"favorable","non_favorable","insuffisant"})
display(pd.DataFrame(checks))

## 3. Description du jeu complet

In [ ]:
jeu_complet=jeu_source.copy(); y_true=jeu_complet.true_label; y_pred=jeu_complet.classe_cnn_top1
cnn_accuracy=accuracy_score(y_true,y_pred); cnn_macro_f1=f1_score(y_true,y_pred,average="macro")
print(f"Accuracy CNN: {cnn_accuracy:.3%} — macro-F1: {cnn_macro_f1:.3f}")
fig,axes=plt.subplots(1,3,figsize=(18,5))
y_true.value_counts().sort_values().plot.barh(ax=axes[0],title="Classes réelles")
y_pred.value_counts().sort_values().plot.barh(ax=axes[1],title="Classes Top-1")
jeu_complet.score_cnn_top1.plot.hist(bins=30,ax=axes[2],title="Score Top-1")
plt.tight_layout()
labels=sorted(set(y_true)|set(y_pred)); cm=confusion_matrix(y_true,y_pred,labels=labels)
plt.figure(figsize=(11,9));sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, linewidths=.4, linecolor="white", cbar_kws={"label":"Nombre d’images"})
plt.title("Matrice de confusion CNN — effectifs absolus");plt.xlabel("Prédiction");plt.ylabel("Classe réelle");plt.tight_layout()
class_perf=pd.DataFrame({"n":y_true.value_counts(),"rappel":pd.Series({x:recall_score(y_true==x,y_pred==x) for x in labels}),"F1":pd.Series({x:f1_score(y_true==x,y_pred==x) for x in labels})})
display(class_perf.sort_index())
display(jeu_complet.groupby(["region","date_observation"]).size().rename("images").reset_index().head(20))
display(jeu_complet.bulletin_id.fillna("sans_bulletin").value_counts().head(20).to_frame("images"))

## 4. Vérification du protocole expérimental

In [ ]:
protocol=(jeu_complet.groupby("scenario_assigne").agg(demande=("image_id","size"),realisable=("scenario_realisable","sum")).astype(int))
protocol["pourcentage_demande"]=protocol.demande/len(jeu_complet)
protocol["taux_realisation"]=protocol.realisable/protocol.demande
display(protocol)
expected={"favorable":(1505,695),"non_favorable":(1504,679),"insuffisant":(752,752)}
for scenario,(n,r) in expected.items(): assert tuple(protocol.loc[scenario,["demande","realisable"]])==(n,r)
print("Tableau attendu vérifié — OK")
display(pd.crosstab(jeu_complet.scenario_assigne,jeu_complet.scenario_realise,normalize="index").style.format("{:.1%}"))
display(jeu_complet.loc[~jeu_complet.scenario_realisable,"raison_non_realisation"].fillna("non précisée").value_counts().to_frame("n"))
display(jeu_complet.groupby(["true_label","scenario_assigne"]).scenario_realisable.mean().unstack().style.format("{:.1%}"))

## 5. Définition explicite des populations

Le filtre du jeu principal est affiché directement ci-dessous.

In [ ]:
classes_couvertes=jeu_complet.loc[jeu_complet.classe_couverte_bsv.astype(bool)].copy()
jeu_realisable=classes_couvertes.loc[classes_couvertes.scenario_realisable.astype(bool)].copy()
jeu_decidable=jeu_realisable.loc[jeu_realisable.verdict_bsv_top1.isin(["Coherent","Non_corrobore"])].copy()
# Filtre principal explicite: classe couverte ET scénario réalisable ET verdict décidable.
jeu_principal=jeu_complet.loc[
 jeu_complet.classe_couverte_bsv.astype(bool)
 & jeu_complet.scenario_realisable.astype(bool)
 & jeu_complet.verdict_bsv_top1.isin(["Coherent","Non_corrobore"])
].copy()
assert len(jeu_principal)==1246
populations={"jeu_complet":jeu_complet,"classes_couvertes":classes_couvertes,"jeu_realisable":jeu_realisable,"jeu_decidable":jeu_decidable,"jeu_principal":jeu_principal}
rows=[]
for name,d in populations.items():
 rows.append({"population":name,"images":len(d),"classes":d.true_label.nunique(),"groupes":d.bulletin_id.fillna(d.context_id).nunique(),"accuracy_CNN":d.cnn_correct.mean(),"scénarios":d.scenario_realise.value_counts().to_dict(),"verdicts":d.verdict_bsv_top1.value_counts().to_dict()})
display(pd.DataFrame(rows))

## 6. Analyse descriptive du moteur

Cette section évalue sa couverture et son comportement, sans conclure sur son apport au CNN.

In [ ]:
desc=[]
for name,d in populations.items():
 decision=d.verdict_bsv_top1.isin(["Coherent","Non_corrobore"])
 desc.append({"population":name,"n":len(d),"taux_decision":decision.mean(),"taux_indetermine":1-decision.mean(),"verdicts":d.verdict_bsv_top1.value_counts().to_dict()})
display(pd.DataFrame(desc))
indet=jeu_complet.loc[jeu_complet.verdict_bsv_top1=="Indetermine"]
display(indet.niveau_preuve.fillna("manquant").value_counts().to_frame("causes d’indétermination"))
for col in ["scenario_realise","true_label","classe_cnn_top1","region"]:
 display(pd.crosstab(jeu_complet[col],jeu_complet.verdict_bsv_top1,normalize="index").style.format("{:.1%}"))
display(pd.crosstab(pd.qcut(jeu_complet.score_cnn_top1,10,duplicates="drop"),jeu_complet.verdict_bsv_top1,normalize="index").style.format("{:.1%}"))
display(pd.crosstab(jeu_complet.bulletin_id.fillna(jeu_complet.context_id),jeu_complet.verdict_bsv_top1).head(20))

## 7. Capacité à retrouver le scénario construit

La correspondance binaire est définie sur les scénarios `favorable` et `non_favorable`. Le scénario construit reste un élément expérimental, non une vérité clinique indépendante.

In [ ]:
recovery=jeu_complet.loc[jeu_complet.scenario_realisable & jeu_complet.scenario_realise.isin(["favorable","non_favorable"]) & jeu_complet.verdict_bsv_top1.isin(["Coherent","Non_corrobore"])].copy()
recovery["scenario_binary"]=(recovery.scenario_realise=="favorable").astype(int)
recovery["verdict_binary"]=(recovery.verdict_bsv_top1=="Coherent").astype(int)
def binary_metrics(y,p):
 tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
 return {"accuracy":accuracy_score(y,p),"balanced_accuracy":balanced_accuracy_score(y,p),"sensibilite":tp/(tp+fn),"specificite":tn/(tn+fp),"precision":precision_score(y,p,zero_division=0),"rappel":recall_score(y,p),"F1":f1_score(y,p),"MCC":matthews_corrcoef(y,p)}
recovery_metrics=binary_metrics(recovery.scenario_binary,recovery.verdict_binary)
display(pd.crosstab(recovery.scenario_realise,recovery.verdict_bsv_top1));display(pd.DataFrame([recovery_metrics]))
# IC bootstrap groupé des métriques de récupération.
rng=np.random.default_rng(SEED); recovery["group"]=recovery.bulletin_id.fillna(recovery.context_id).astype(str)
groups=recovery.group.unique(); boot_rec=[]
for _ in range(BOOTSTRAP_REPLICATIONS):
 chosen=rng.choice(groups,len(groups),replace=True); sample=pd.concat([recovery.loc[recovery.group==g] for g in chosen],ignore_index=True)
 if sample.scenario_binary.nunique()==2: boot_rec.append(binary_metrics(sample.scenario_binary,sample.verdict_binary))
recovery_ci=pd.DataFrame(boot_rec).quantile([.025,.5,.975]).T;recovery_ci.columns=["IC95_bas","médiane","IC95_haut"]
display(recovery_ci)

## 8. Association brute entre verdict et justesse du CNN

In [ ]:
observed=pd.crosstab(jeu_principal.cnn_correct,jeu_principal.verdict_bsv_top1).reindex(index=[True,False],columns=["Coherent","Non_corrobore"],fill_value=0)
stat,pval,_,expected_counts=chi2_contingency(observed,correction=False);phi=np.sqrt(stat/observed.values.sum())
a,b=observed.loc[True];c_,d=observed.loc[False]; raw_or=a*d/(b*c_);se=np.sqrt(1/a+1/b+1/c_+1/d)
display(observed);display(pd.DataFrame(expected_counts,index=observed.index,columns=observed.columns))
display(pd.DataFrame([{"chi2":stat,"p_value":pval,"phi":phi,"OR_brut":raw_or,"IC95_bas":np.exp(np.log(raw_or)-1.96*se),"IC95_haut":np.exp(np.log(raw_or)+1.96*se),"taux_coherent_correct":a/(a+b),"taux_coherent_incorrect":c_/(c_+d)}]))

## 9. Analyse principale : modèles emboîtés

In [ ]:
model=jeu_principal[["cnn_correct","score_cnn_top1","verdict_bsv_top1","bulletin_id","context_id"]].copy()
assert not model[["cnn_correct","score_cnn_top1","verdict_bsv_top1"]].isna().any().any()
model["Y"]=model.cnn_correct.astype(int);model["V"]=(model.verdict_bsv_top1=="Coherent").astype(int)
model["score_borne"]=model.score_cnn_top1.clip(1e-6,1-1e-6);model["logit_score"]=np.log(model.score_borne/(1-model.score_borne))
model["group"]=model.bulletin_id.fillna(model.context_id).astype(str)
print(f"Groupes: {model.group.nunique()} — erreurs: {(model.Y==0).sum()} — groupes avec erreur: {model.loc[model.Y==0,'group'].nunique()}")
display(pd.crosstab(model.group,model.Y).sort_values(0,ascending=False).head(15))
# Linéarité empirique.
model["score_bin"]=pd.qcut(model.score_borne,10,duplicates="drop")
lin=model.groupby("score_bin",observed=True).agg(n=("Y","size"),correct=("Y","sum"),mean_logit=("logit_score","mean"));lin["emp_logodds"]=np.log((lin.correct+.5)/(lin.n-lin.correct+.5))
plt.figure(figsize=(7,4));plt.scatter(lin.mean_logit,lin.emp_logodds,s=lin.n);plt.plot(lin.mean_logit,lin.emp_logodds);plt.xlabel("logit(score)");plt.ylabel("log-odds empiriques");plt.title("Contrôle de linéarité");plt.tight_layout()
print("Cellules verdict × issue:");display(pd.crosstab(model.V,model.Y))
X0=sm.add_constant(model[["logit_score"]]);X1=sm.add_constant(model[["logit_score","V"]])
M0=sm.GLM(model.Y,X0,family=sm.families.Binomial()).fit();M1=sm.GLM(model.Y,X1,family=sm.families.Binomial()).fit()
M1_cluster=sm.GLM(model.Y,X1,family=sm.families.Binomial()).fit(cov_type="cluster",cov_kwds={"groups":model.group})
lr=2*(M1.llf-M0.llf);beta=M1.params.V;se_cluster=M1_cluster.bse.V
main_result=pd.DataFrame([{"LRT":lr,"p_LRT":chi2.sf(lr,1),"beta_verdict":beta,"OR_ajuste":np.exp(beta),"IC95_conventionnel_bas":np.exp(beta-1.96*M1.bse.V),"IC95_conventionnel_haut":np.exp(beta+1.96*M1.bse.V),"z_Wald_cluster":beta/se_cluster,"p_Wald_cluster":2*__import__('scipy').stats.norm.sf(abs(beta/se_cluster)),"IC95_cluster_bas":np.exp(beta-1.96*se_cluster),"IC95_cluster_haut":np.exp(beta+1.96*se_cluster)}]);display(main_result)
# Bootstrap groupé de l'OR.
rng=np.random.default_rng(SEED);groups=model.group.unique();boot_beta=[]
for _ in range(BOOTSTRAP_REPLICATIONS):
 chosen=rng.choice(groups,len(groups),replace=True);s=pd.concat([model.loc[model.group==g] for g in chosen],ignore_index=True)
 try: boot_beta.append(sm.GLM(s.Y,sm.add_constant(s[["logit_score","V"]]),family=sm.families.Binomial()).fit(disp=0).params.V)
 except Exception: pass
boot_or=np.exp(boot_beta);display(pd.DataFrame([{"réplications_réussies":len(boot_or),"groupes":len(groups),"OR_médian":np.median(boot_or),"IC95_boot_bas":np.quantile(boot_or,.025),"IC95_boot_haut":np.quantile(boot_or,.975)}]))

## 10. Validation croisée groupée

In [ ]:
features={"M0":["logit_score"],"M1":["logit_score","V"]};gkf=GroupKFold(n_splits=CV_FOLDS);cv=[]
for fold,(tr,te) in enumerate(gkf.split(model,model.Y,model.group),1):
 for name,cols in features.items():
  clf=LogisticRegression(penalty=None,max_iter=2000).fit(model.iloc[tr][cols],model.Y.iloc[tr]);prob=clf.predict_proba(model.iloc[te][cols])[:,1];y=model.Y.iloc[te]
  cv.append({"pli":fold,"modèle":name,"n_test":len(te),"log_loss":log_loss(y,prob),"Brier":brier_score_loss(y,prob),"ROC_AUC":roc_auc_score(y,prob),"PR_AUC_erreurs":average_precision_score(1-y,1-prob)})
cv=pd.DataFrame(cv);display(cv)
summary=cv.groupby("modèle").agg(["mean","std"])[["log_loss","Brier","ROC_AUC","PR_AUC_erreurs"]];display(summary)
paired=cv.pivot(index="pli",columns="modèle",values=["log_loss","Brier","ROC_AUC","PR_AUC_erreurs"])
deltas=pd.DataFrame({metric:paired[metric]["M1"]-paired[metric]["M0"] for metric in paired.columns.levels[0]});display(deltas.agg(["mean","std"]).T.rename(columns={"mean":"différence_M1-M0","std":"dispersion"}))

## 11. Analyses de sensibilité

In [ ]:
def sensitivity_fit(label,d,group_col="bulletin_id",score_form="logit"):
 z=d.loc[d.verdict_bsv_top1.isin(["Coherent","Non_corrobore"])].copy();z["Y"]=z.cnn_correct.astype(int);z["V"]=(z.verdict_bsv_top1=="Coherent").astype(int);z["score"]=z.score_cnn_top1.clip(1e-6,1-1e-6);z["x"]=np.log(z.score/(1-z.score)) if score_form=="logit" else z.score
 if len(z)<30 or z.Y.nunique()<2:return {"analyse":label,"n":len(z),"statut":"insuffisant"}
 X=sm.add_constant(z[["x","V"]]);fit=sm.GLM(z.Y,X,family=sm.families.Binomial()).fit();groups=z[group_col].fillna(z.context_id).astype(str);rob=sm.GLM(z.Y,X,family=sm.families.Binomial()).fit(cov_type="cluster",cov_kwds={"groups":groups})
 return {"analyse":label,"n":len(z),"erreurs":int((z.Y==0).sum()),"groupes":groups.nunique(),"OR":np.exp(fit.params.V),"IC_bas":np.exp(fit.params.V-1.96*rob.bse.V),"IC_haut":np.exp(fit.params.V+1.96*rob.bse.V),"p_cluster":rob.pvalues.V}
sens=[sensitivity_fit("principal",jeu_principal),sensitivity_fit("sans healthy",jeu_principal.loc[~jeu_principal.true_label.str.contains("healthy",case=False)]),sensitivity_fit("groupes context_id",jeu_principal,"context_id"),sensitivity_fit("score brut",jeu_principal,score_form="raw")]
for key,d in jeu_principal.groupby("scenario_realise"):sens.append(sensitivity_fit(f"scénario {key}",d))
for key,d in jeu_principal.groupby("region"):
 if len(d)>=40:sens.append(sensitivity_fit(f"région {key}",d))
# Exclusion des groupes de moins de 3 observations.
g=jeu_principal.bulletin_id.fillna(jeu_principal.context_id);keep=g.map(g.value_counts()).ge(3);sens.append(sensitivity_fit("groupes >= 3 observations",jeu_principal.loc[keep]))
display(pd.DataFrame(sens))
# Spline du logit(score), avec effet du verdict.
spline=np.asarray(dmatrix("bs(x, df=4, include_intercept=False)",{"x":model.logit_score}));Xs=np.column_stack([spline,model.V]);fit_spline=sm.GLM(model.Y,Xs,family=sm.families.Binomial()).fit();display(pd.DataFrame([{"analyse":"spline logit(score)","OR_verdict":np.exp(fit_spline.params[-1]),"beta_verdict":fit_spline.params[-1]}]))

## 12. Sensibilité décisive avec contrôle de la classe prédite

Cette analyse complète M0–M1 en contrôlant la classe prédite :

\[
M_C : Y \sim \operatorname{logit}(score)+classe_{prédite}
\]

\[
M_2 : Y \sim \operatorname{logit}(score)+classe_{prédite}+verdict
\]

La ligne essentielle est **« MC vs M2 : ajout du verdict »**. Elle teste si le verdict du moteur fondé sur le BSV apporte encore une information après contrôle du score CNN et de la classe prédite. Aucune interaction `verdict × classe` n’est ajustée.

In [ ]:
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import norm

# Diagnostic par classe prédite, centré sur le nombre d'erreurs.
class_rows=[]
for predicted_class,frame in jeu_principal.groupby("classe_cnn_top1",sort=True):
    errors=frame.loc[~frame.cnn_correct.astype(bool)]
    error_counts=errors.verdict_bsv_top1.value_counts()
    correct=int(frame.cnn_correct.sum());incorrect=len(frame)-correct
    coherent_errors=int(error_counts.get("Coherent",0))
    non_corrobore_errors=int(error_counts.get("Non_corrobore",0))
    class_rows.append({
        "classe_prédite":predicted_class,"observations":len(frame),
        "correctes":correct,"erreurs":incorrect,
        "erreurs_cohérentes":coherent_errors,
        "erreurs_non_corroborees":non_corrobore_errors,
        "cellule_issue_nulle":correct==0 or incorrect==0,
        "cellule_erreur_verdict_nulle":coherent_errors==0 or non_corrobore_errors==0,
        "cellule_presque_nulle":min(correct,incorrect,coherent_errors,non_corrobore_errors)<5,
    })
class_diagnostic_notebook=pd.DataFrame(class_rows)
display(class_diagnostic_notebook)

# Tableau classe × verdict × justesse demandé.
class_verdict_correctness=(jeu_principal.groupby(
    ["classe_cnn_top1","verdict_bsv_top1","cnn_correct"],observed=True
).size().rename("effectif").reset_index())
display(class_verdict_correctness)
display(class_verdict_correctness.pivot_table(
    index="classe_cnn_top1",columns=["verdict_bsv_top1","cnn_correct"],
    values="effectif",fill_value=0
))

separation_classes=class_diagnostic_notebook.loc[
    class_diagnostic_notebook.cellule_issue_nulle,"classe_prédite"
].tolist()
firth_required=bool(separation_classes)
print(f"Classes avec séparation de l'issue : {separation_classes or 'aucune'}")
print(f"Régression de Firth nécessaire : {firth_required}")

# Matrices MC, M2 et M1. La première classe alphabétique est la référence.
class_model=jeu_principal[["cnn_correct","score_cnn_top1","verdict_bsv_top1",
                           "classe_cnn_top1","bulletin_id","context_id"]].copy().reset_index(drop=True)
class_model["Y"]=class_model.cnn_correct.astype(int)
class_model["V"]=(class_model.verdict_bsv_top1=="Coherent").astype(int)
class_model["score_borne"]=class_model.score_cnn_top1.clip(1e-6,1-1e-6)
class_model["logit_score"]=np.log(class_model.score_borne/(1-class_model.score_borne))
class_model["group"]=class_model.bulletin_id.fillna(class_model.context_id).astype(str)
dummies=pd.get_dummies(class_model.classe_cnn_top1,prefix="classe",drop_first=True,dtype=float)
y_class=class_model.Y.to_numpy(float)
X_MC=np.column_stack([np.ones(len(class_model)),class_model.logit_score,dummies.to_numpy()])
X_M2=np.column_stack([X_MC,class_model.V])
X_M1=np.column_stack([np.ones(len(class_model)),class_model.logit_score,class_model.V])
terms_MC=["intercept","logit_score",*dummies.columns.tolist()]
terms_M2=[*terms_MC,"verdict_coherent"]
parameter_dimension=pd.DataFrame([
    {"modèle":"MC","paramètres_avec_intercept":X_MC.shape[1],"variables_hors_intercept":X_MC.shape[1]-1,"erreurs_par_variable":(y_class==0).sum()/(X_MC.shape[1]-1)},
    {"modèle":"M2","paramètres_avec_intercept":X_M2.shape[1],"variables_hors_intercept":X_M2.shape[1]-1,"erreurs_par_variable":(y_class==0).sum()/(X_M2.shape[1]-1)},
])
display(parameter_dimension)

# Logit de Firth : pénalisation de Jeffreys, activée uniquement car une
# séparation concrète a été détectée dans les classes prédites.
def fit_firth_notebook(y,x):
    def objective(beta):
        eta=x@beta;p=expit(eta)
        ll=float(y@eta-np.logaddexp(0,eta).sum())
        info=x.T@(x*(p*(1-p))[:,None])
        sign,logdet=np.linalg.slogdet(info)
        return -(ll+.5*logdet) if sign>0 and np.isfinite(logdet) else 1e100
    result=minimize(objective,np.zeros(x.shape[1]),method="BFGS",options={"gtol":1e-7,"maxiter":2000})
    beta=np.asarray(result.x);p=expit(x@beta);info=x.T@(x*(p*(1-p))[:,None])
    if not np.isfinite(beta).all() or not np.isfinite(result.fun):
        raise RuntimeError(f"Échec Firth: {result.message}")
    return {"beta":beta,"penalized_ll":-float(result.fun),"probability":p,"information":info,"covariance":np.linalg.pinv(info),"message":str(result.message)}

def fit_standard_notebook(y,x):
    fit=sm.GLM(y,x,family=sm.families.Binomial()).fit()
    p=np.asarray(fit.fittedvalues);info=x.T@(x*(p*(1-p))[:,None])
    return {"beta":np.asarray(fit.params),"penalized_ll":float(fit.llf),"probability":p,"information":info,"covariance":np.asarray(fit.cov_params()),"message":"MLE convergée"}

fit_function=fit_firth_notebook if firth_required else fit_standard_notebook
estimation_method="Firth" if firth_required else "Maximum de vraisemblance"
fit_MC=fit_function(y_class,X_MC);fit_M2=fit_function(y_class,X_M2);fit_M1_class=fit_function(y_class,X_M1)

# Covariance sandwich groupée, évaluée au coefficient M2. Pour Firth, il
# s'agit d'un diagnostic robuste sandwich autour de l'estimation pénalisée.
def cluster_covariance_notebook(y,x,fit,groups):
    unique=pd.unique(groups);residual=y-fit["probability"];meat=np.zeros((x.shape[1],x.shape[1]))
    for group in unique:
        score=x[groups==group].T@residual[groups==group];meat+=np.outer(score,score)
    bread=np.linalg.pinv(fit["information"]);n,p=x.shape
    correction=(len(unique)/(len(unique)-1))*((n-1)/(n-p))
    return correction*bread@meat@bread

cluster_cov_M2=cluster_covariance_notebook(y_class,X_M2,fit_M2,class_model.group.to_numpy())
verdict_index=len(terms_M2)-1;beta_verdict=float(fit_M2["beta"][verdict_index])
cluster_se=float(np.sqrt(cluster_cov_M2[verdict_index,verdict_index]));wald_z=beta_verdict/cluster_se
lr_MC_M2=2*(fit_M2["penalized_ll"]-fit_MC["penalized_ll"])
lr_M1_M2=2*(fit_M2["penalized_ll"]-fit_M1_class["penalized_ll"])
class_adjusted_comparison=pd.DataFrame([
    {"comparaison":"MC vs M2 : ajout du verdict","ajout_testé":"verdict","méthode":estimation_method,"ddl":1,"LRT":max(lr_MC_M2,0),"p_LRT":chi2.sf(max(lr_MC_M2,0),1)},
    {"comparaison":"M1 vs M2 : ajout global de la classe","ajout_testé":"classe prédite","méthode":estimation_method,"ddl":dummies.shape[1],"LRT":max(lr_M1_M2,0),"p_LRT":chi2.sf(max(lr_M1_M2,0),dummies.shape[1])},
])
display(class_adjusted_comparison)

class_adjusted_result=pd.DataFrame([{
    "méthode":estimation_method,"observations":len(class_model),"erreurs":int((y_class==0).sum()),
    "classes_prédites":class_model.classe_cnn_top1.nunique(),"groupes_bulletin":class_model.group.nunique(),
    "coefficient_verdict_M2":beta_verdict,"OR_verdict_ajusté_score_classe":np.exp(beta_verdict),
    "SE_cluster":cluster_se,"z_Wald_cluster":wald_z,"p_Wald_cluster":2*norm.sf(abs(wald_z)),
    "IC95_OR_cluster_bas":np.exp(beta_verdict-1.96*cluster_se),
    "IC95_OR_cluster_haut":np.exp(beta_verdict+1.96*cluster_se),
}])
display(class_adjusted_result)

firth_diagnostic_notebook=pd.DataFrame([{
    "Firth_nécessaire":firth_required,"raison":"séparation par classe prédite" if firth_required else "aucune séparation observée",
    "classes_déclenchantes":"; ".join(separation_classes),"convergence_MC":fit_MC["message"],"convergence_M2":fit_M2["message"],
    "coefficient_max_absolu_M2":float(np.max(np.abs(fit_M2["beta"]))),
    "erreur_standard_max_M2":float(np.sqrt(np.diag(fit_M2["covariance"])).max()),
    "probabilité_min_M2":float(fit_M2["probability"].min()),"probabilité_max_M2":float(fit_M2["probability"].max()),
    "interaction_verdict_classe":False,"raison_absence_interaction":"76 erreurs seulement"
}])
display(firth_diagnostic_notebook)

## 13. Analyse qualitative des erreurs

In [ ]:
interpret={(True,"Coherent"):"CNN correct + moteur cohérent",(True,"Non_corrobore"):"CNN correct + moteur non corroboré",(False,"Coherent"):"CNN incorrect + moteur cohérent",(False,"Non_corrobore"):"CNN incorrect + moteur non corroboré"}
cols=["image_id","true_label","classe_cnn_top1","score_cnn_top1","scenario_realise","date_observation","region","bulletin_id","agent_present_bulletin","niveau_preuve","niveau_risque","meteo_favorable","decision_reason"]
examples=[]
for (correct,verdict),label in interpret.items():
 e=jeu_principal.loc[(jeu_principal.cnn_correct==correct)&(jeu_principal.verdict_bsv_top1==verdict),cols].head(5).copy();e.insert(0,"situation",label);examples.append(e)
display(pd.concat(examples,ignore_index=True))

## 14. Conclusion générée à partir des résultats

In [ ]:
coverage_rate=jeu_complet.verdict_bsv_top1.isin(["Coherent","Non_corrobore"]).mean()
scenario_bal=recovery_metrics["balanced_accuracy"]
association_p=pval
incremental_p=main_result.loc[0,"p_Wald_cluster"]
cv_means=cv.groupby("modèle")[["log_loss","Brier"]].mean()
gain=(cv_means.loc["M1"]<cv_means.loc["M0"]).all()
decisive=class_adjusted_comparison.loc[
    class_adjusted_comparison.comparaison=="MC vs M2 : ajout du verdict"
].iloc[0]
decisive_wald_p=float(class_adjusted_result.loc[0,"p_Wald_cluster"])
decisive_or=float(class_adjusted_result.loc[0,"OR_verdict_ajusté_score_classe"])

print(f"1. Couverture : {coverage_rate:.1%} des cas ont un verdict décidable.")
print(f"2. Scénario construit : balanced accuracy = {scenario_bal:.3f}. Ce scénario est expérimental, non une vérité biologique indépendante.")
print(f"3. Association verdict–justesse : χ² p = {association_p:.3g}, φ = {phi:.3f}.")
if incremental_p<.05 and gain:
    print("4. Analyse principale M0–M1 : signal complémentaire robuste et amélioration des deux métriques hors échantillon.")
elif incremental_p<.05:
    print("4. Analyse principale M0–M1 : effet robuste, mais gain prédictif non uniforme.")
else:
    print("4. Analyse principale M0–M1 : apport non démontré après clusterisation.")
print(f"5. Analyse décisive MC–M2 : LRT p = {decisive.p_LRT:.3g}; Wald clusterisé p = {decisive_wald_p:.3g}; OR ajusté score + classe = {decisive_or:.2f}.")
if decisive.p_LRT<.05 and decisive_wald_p<.05:
    print("Conclusion ajustée sur la classe : le verdict du moteur fondé sur le BSV apporte encore une information après contrôle du score CNN et de la classe prédite.")
else:
    print("Conclusion ajustée sur la classe : l'apport supplémentaire du verdict n'est pas démontré de manière robuste après contrôle de la classe prédite.")
print(f"Méthode MC/M2 : {estimation_method}; Firth nécessaire = {firth_required}; interaction verdict × classe non estimée.")
print("Limite : les contextes ont été construits à partir de la vraie classe; la généralisation à des contextes BSV réels attribués indépendamment reste à démontrer.")